# 🩺 Diabetic Retinopathy Grading — Elite Pipeline v18
## Regression QWK + 2-Stage Ordinal | 5-Fold CV | Ensemble | TTA | Grad-CAM++
### ✅ Colab · Kaggle · Windows · Linux — Full Auto-Download Included

---

**What's new in v18 (merged from community regression approach):**
- **Regression backbone** (`num_classes=1`, MSE loss) — mathematically optimal for QWK  
- **OptimizedRounder** via Nelder-Mead — fits thresholds [t0,t1,t2,t3] directly on val predictions  
- **Trainer class** — clean encapsulation of train/validate/checkpoint  
- **Phase-aware freeze** — per-phase `freeze` flag controls backbone parameters  
- **Early stopping** per phase with configurable patience  
- **AMP (mixed precision)** throughout — faster + lower VRAM  
- **WeightedRandomSampler** — correct oversampling on every loader  
- **Full resume** — saves epoch, batch, phase, optimizer, scheduler, best_qwk  

**Run order:**  
`Cell 1` → `Cell 2` → `Cell 2b (Download)` → `Cell 3` → `Cell 4` → `Cell 5` → `Cell 6` → `Cell 7` → `Cell 8` → `Cell 9` → `Cell 10` → `Cell 11 (Train)` → `Cell 12 (Threshold)` → `Cell 13 (Eval)` → `Cell 14 (Ensemble)` → `Cell 15 (2-Stage)` → `Cell 16 (Grad-CAM)` → `Cell 17 (Summary)`


## 📦 Cell 1 — Install Dependencies

In [ ]:
# ── Cell 1: Safe package installer ────────────────────────────────────────────
import sys
import subprocess

PACKAGES = [
    "timm>=1.0.3",
    "albumentations>=1.4.0,<2.0.0",
    "opencv-python-headless",
    "scikit-learn",
    "pandas",
    "numpy",
    "tqdm",
    "matplotlib",
    "scipy",
    "pytorch-grad-cam",   # correct PyPI package name (was: grad-cam)
    "pyarrow",
    "fastparquet",
    "kaggle",
    "packaging",
]

print("Installing / upgrading packages ...")
failed = []
for pkg in PACKAGES:
    print(f"  {pkg} ...", end=" ", flush=True)
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade", pkg],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print("❌")
        print(result.stderr[-600:])
        failed.append(pkg)
    else:
        print("✅")

if failed:
    raise RuntimeError(f"These packages failed to install: {failed}")
print("\n✅ All packages installed successfully.")


## ⚙️ Cell 2 — Imports, Config & Device

In [ ]:
import os, sys, io, json, gc, time, random, shutil, warnings, zipfile
from pathlib import Path
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from tqdm.auto import tqdm
from packaging.version import Version

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    cohen_kappa_score, accuracy_score,
    confusion_matrix, classification_report,
    roc_auc_score, ConfusionMatrixDisplay,
)
from scipy.optimize import minimize

warnings.filterwarnings("ignore")

# ── Seed ─────────────────────────────────────────────────────────────────────
CFG = {
    "seed"          : 42,
    "model_name"    : "tf_efficientnetv2_b1",
    "n_folds"       : 5,
    "lr"            : 3e-4,
    "min_lr"        : 1e-6,
    "weight_decay"  : 1e-4,
    "patience"      : 5,
    "grad_clip"     : 1.0,
    "phases": [
        {"id": 1, "size": 224, "batch_size": 32, "epochs": 15, "freeze": True},
        {"id": 2, "size": 384, "batch_size": 16, "epochs": 30, "freeze": False},
        {"id": 3, "size": 512, "batch_size":  8, "epochs": 20, "freeze": False},
    ],
}

def seed_everything(seed=CFG["seed"]):
    random.seed(seed); np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
seed_everything()

# ── Device ───────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"🔥 GPU: {torch.cuda.get_device_name(0)}"
          f"  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB"
          f"  CUDA: {torch.version.cuda}")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("🍎 Apple MPS device detected.")
else:
    DEVICE = torch.device("cpu")
    print("💻 No GPU found — CPU mode (slow).")

USE_AMP = (DEVICE.type == "cuda")

# ── Environment detection ─────────────────────────────────────────────────────
IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/input")
IN_WIN    = sys.platform == "win32"

if IN_KAGGLE:
    DATA_DIR     = Path("/kaggle/input/aptos2019-blindness-detection")
    ARTIFACT_DIR = Path("/kaggle/working/artifacts_v18")
elif IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception:
        pass
    DATA_DIR     = Path("/content/drive/MyDrive/DR_data/aptos2019")
    ARTIFACT_DIR = Path("/content/drive/MyDrive/DR_data/artifacts_v18")
else:
    DATA_DIR     = Path(os.environ.get("DATA_DIR",     str(Path.home()/"DR_data"/"aptos2019")))
    ARTIFACT_DIR = Path(os.environ.get("ARTIFACT_DIR", str(Path.home()/"DR_data"/"artifacts_v18")))

IMG_DIR  = DATA_DIR / "train_images"
CSV_PATH = DATA_DIR / "train.csv"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR = ARTIFACT_DIR / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

NUM_CLASSES  = 5
GRADE_MAP    = {0:"No DR", 1:"Mild", 2:"Moderate", 3:"Severe", 4:"Proliferative"}
GRADE_COLORS = ["#2ecc71","#f1c40f","#e67e22","#e74c3c","#8e44ad"]
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ── State helpers ─────────────────────────────────────────────────────────────
_STATE = ARTIFACT_DIR / "state_v18.json"
def st_load():
    return json.loads(_STATE.read_text()) if _STATE.exists() else {}
def st_save(k, v):
    s = st_load(); s[k] = v; _STATE.write_text(json.dumps(s, indent=2))

def safe_load(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

print(f"\n✅ Imports complete | PyTorch {torch.__version__} | timm {timm.__version__}")
print(f"   Device: {DEVICE} | AMP: {'ON' if USE_AMP else 'OFF'}")
print(f"   DATA_DIR:     {DATA_DIR}")
print(f"   ARTIFACT_DIR: {ARTIFACT_DIR}")


## 📥 Cell 2b — Dataset Download (APTOS 2019)
Automatically downloads **aptos2019-blindness-detection** from Kaggle.

| Environment | Method |
|-------------|--------|
| **Kaggle Notebook** | Data already mounted at `/kaggle/input/` — skipped automatically |
| **Google Colab** | Upload `kaggle.json` when prompted, then downloads & unzips |
| **Windows / Linux local** | Place `kaggle.json` in `~/.kaggle/` (or set `KAGGLE_USERNAME` + `KAGGLE_KEY` env vars) |

> **Get your kaggle.json**: kaggle.com → Account → API → "Create New Token"


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 2b — APTOS 2019 Dataset Download                          ║
# ║  Safe to re-run: skips download if data already present         ║
# ╚══════════════════════════════════════════════════════════════════╝

import os, sys, zipfile, shutil, subprocess
from pathlib import Path

# ── These were already defined in Cell 2 ─────────────────────────────────────
try:
    _ = DATA_DIR
except NameError:
    IN_COLAB  = "google.colab" in sys.modules
    IN_KAGGLE = os.path.exists("/kaggle/input")
    if IN_KAGGLE:
        DATA_DIR = Path("/kaggle/input/aptos2019-blindness-detection")
    elif IN_COLAB:
        DATA_DIR = Path("/content/drive/MyDrive/DR_data/aptos2019")
    else:
        DATA_DIR = Path.home() / "DR_data" / "aptos2019"
    IMG_DIR  = DATA_DIR / "train_images"
    CSV_PATH = DATA_DIR / "train.csv"

COMPETITION = "aptos2019-blindness-detection"

def _pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

def _check_complete():
    if not CSV_PATH.exists():
        return False
    n_imgs = len(list(IMG_DIR.glob("*.png"))) if IMG_DIR.exists() else 0
    return n_imgs >= 3000

# ── Already downloaded? ───────────────────────────────────────────────────────
if _check_complete():
    n = len(list(IMG_DIR.glob("*.png")))
    print(f"✅ Dataset already present: {CSV_PATH.parent}")
    print(f"   train.csv ✓  |  {n} train images ✓")

# ── Kaggle Notebook: data is auto-mounted ────────────────────────────────────
elif IN_KAGGLE:
    src = Path(f"/kaggle/input/{COMPETITION}")
    if src.exists():
        DATA_DIR.parent.mkdir(parents=True, exist_ok=True)
        if not DATA_DIR.exists():
            DATA_DIR.symlink_to(src)
        print(f"✅ Kaggle: data symlinked from {src} → {DATA_DIR}")
    else:
        print(f"⚠️  In Kaggle but competition data not found at {src}")
        print(f"   Add '{COMPETITION}' as a Dataset in the right panel.")

# ── Google Colab ──────────────────────────────────────────────────────────────
elif IN_COLAB:
    _pip_install("kaggle")
    from google.colab import files as _colab_files

    kaggle_cfg = Path.home() / ".kaggle" / "kaggle.json"
    if not kaggle_cfg.exists():
        print("📤 Upload your kaggle.json (kaggle.com → Account → API → Create New Token)")
        _uploaded = _colab_files.upload()
        kaggle_cfg.parent.mkdir(parents=True, exist_ok=True)
        for fname, data in _uploaded.items():
            kaggle_cfg.write_bytes(data)
        kaggle_cfg.chmod(0o600)
        print(f"✅ kaggle.json saved → {kaggle_cfg}")

    DATA_DIR.mkdir(parents=True, exist_ok=True)
    _zip = DATA_DIR / f"{COMPETITION}.zip"

    if not _zip.exists():
        print(f"⬇️  Downloading {COMPETITION} (~1.5 GB) ...")
        subprocess.check_call([
            sys.executable, "-m", "kaggle", "competitions", "download",
            "-c", COMPETITION, "-p", str(DATA_DIR)
        ])
        print("✅ Download complete.")

    print("📦 Unzipping ...")
    with zipfile.ZipFile(_zip, "r") as z:
        z.extractall(DATA_DIR)

    for nested in DATA_DIR.glob("*.zip"):
        print(f"   Unzipping nested: {nested.name}")
        with zipfile.ZipFile(nested, "r") as z:
            z.extractall(DATA_DIR)
        nested.unlink()

    _zip.unlink(missing_ok=True)
    n = len(list(IMG_DIR.glob("*.png")))
    print(f"✅ Extraction complete: {n} images in {IMG_DIR}")

# ── Local Windows / Linux ──────────────────────────────────────────────────────
else:
    _pip_install("kaggle")

    kaggle_cfg = Path.home() / ".kaggle" / "kaggle.json"
    env_key    = os.environ.get("KAGGLE_KEY", "")
    env_user   = os.environ.get("KAGGLE_USERNAME", "")

    if not kaggle_cfg.exists() and not (env_key and env_user):
        print("=" * 60)
        print("  ⚙️  KAGGLE API SETUP REQUIRED (one-time)")
        print("=" * 60)
        print("  1. Go to https://www.kaggle.com/settings/account")
        print("  2. Scroll to 'API' section → click 'Create New Token'")
        print("  3. A file 'kaggle.json' will be downloaded")
        print(f"  4. Move it to: {kaggle_cfg.parent}")
        print("     (Windows: C:\\Users\\<YourName>\\.kaggle\\kaggle.json)")
        print()
        print("  OR set environment variables:")
        print("     KAGGLE_USERNAME=<your_username>")
        print("     KAGGLE_KEY=<your_api_key>")
        print("=" * 60)
        raise EnvironmentError("kaggle.json not found. See instructions above.")

    if kaggle_cfg.exists():
        try:
            kaggle_cfg.chmod(0o600)
        except Exception:
            pass

    DATA_DIR.mkdir(parents=True, exist_ok=True)
    _zip = DATA_DIR / f"{COMPETITION}.zip"

    if not _zip.exists():
        print(f"⬇️  Downloading APTOS 2019 (~1.5 GB) to {DATA_DIR} ...")
        print("   This is a one-time download (~3–10 min depending on internet speed)")
        result = subprocess.run([
            sys.executable, "-m", "kaggle", "competitions", "download",
            "-c", COMPETITION, "-p", str(DATA_DIR)
        ], capture_output=True, text=True)
        if result.returncode != 0:
            print("STDERR:", result.stderr)
            raise RuntimeError(f"Kaggle download failed:\n{result.stderr}")
        print("✅ Download complete.")
    else:
        print(f"✅ ZIP already downloaded: {_zip}")

    print("📦 Unzipping main archive ...")
    with zipfile.ZipFile(_zip, "r") as z:
        members = z.namelist()
        print(f"   Contents: {members[:5]} {'...' if len(members)>5 else ''}")
        z.extractall(DATA_DIR)

    for nested in DATA_DIR.glob("*.zip"):
        print(f"📦 Unzipping nested: {nested.name}")
        with zipfile.ZipFile(nested, "r") as z:
            z.extractall(DATA_DIR)
        nested.unlink()

    _zip.unlink(missing_ok=True)
    n = len(list(IMG_DIR.glob("*.png")))
    print(f"✅ Done: {n:,} train images  |  CSV: {CSV_PATH.exists()}")

# ── Final verification ────────────────────────────────────────────────────────
print()
print("─" * 45)
print("DATASET VERIFICATION")
print("─" * 45)
for label, path in [("train.csv", CSV_PATH),
                     ("train_images/", IMG_DIR),
                     ("test_images/",  DATA_DIR/"test_images")]:
    status = "✅" if path.exists() else "❌ MISSING"
    extra  = ""
    if path.exists() and path.is_dir():
        extra = f"  ({len(list(path.glob('*.png')))} .png files)"
    print(f"  {status}  {label}{extra}")
print("─" * 45)

if not _check_complete():
    raise RuntimeError("Dataset incomplete after download. Check errors above.")
print("✅ Dataset ready for training!")


## 📥 Cell 3 — Load & Verify Dataset

In [ ]:
if not CSV_PATH.exists():
    raise FileNotFoundError(f"train.csv missing: {CSV_PATH}\nRun Cell 2b to download the dataset.")

df = pd.read_csv(CSV_PATH)
df["path"]        = df["id_code"].apply(lambda x: str(IMG_DIR / f"{x}.png"))
df["grade_label"] = df["diagnosis"].map(GRADE_MAP)

missing = df[~df["path"].apply(lambda p: Path(p).exists())]
if len(missing):
    print(f"⚠️  Dropping {len(missing)} rows with missing images.")
    df = df[df["path"].apply(lambda p: Path(p).exists())].reset_index(drop=True)

print(f"✅ {len(df):,} images loaded.")
print(f"\nClass distribution  (imbalance: {df.diagnosis.value_counts().max()/df.diagnosis.value_counts().min():.1f}×)")
for g in range(5):
    n   = (df.diagnosis == g).sum()
    bar = "█" * (n // 50)
    print(f"  Grade {g} ({GRADE_MAP[g]:15s}): {n:5d}  {n/len(df)*100:4.1f}%  {bar}")


## 🔬 Cell 4 — Fundus Preprocessing
**Ben Graham method:** crop black borders → resize+pad → CLAHE → `4×img − 4×blur + 128`


In [ ]:
def preprocess_fundus(path_or_img, size=512, sigma_ratio=10):
    """
    Full fundus preprocessing pipeline.
    1. Load BGR → RGB
    2. Auto-crop black borders (mask > 7 threshold)
    3. Pad to square → resize to `size`
    4. Apply circular retinal mask
    5. CLAHE on L channel (LAB)
    6. Ben Graham: 4×img − 4×GaussianBlur + 128
    Returns uint8 RGB [size×size×3]
    """
    if isinstance(path_or_img, np.ndarray):
        img = path_or_img
    else:
        bgr = cv2.imread(str(path_or_img))
        if bgr is None:
            return np.zeros((size, size, 3), np.uint8)
        img = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    # 1. Auto-crop black borders
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    mask = gray > 7
    if mask.any():
        rows = np.where(mask.any(1))[0]; cols = np.where(mask.any(0))[0]
        img  = img[rows[0]:rows[-1]+1, cols[0]:cols[-1]+1]

    # 2. Pad to square
    h, w  = img.shape[:2]
    S     = max(h, w)
    ph    = (S - h) // 2; pb = S - h - ph
    pw    = (S - w) // 2; pr = S - w - pw
    img   = cv2.copyMakeBorder(img, ph, pb, pw, pr, cv2.BORDER_CONSTANT, value=0)

    # 3. Resize
    img = cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)

    # 4. Circular retinal mask
    cmask = np.zeros((size, size), np.uint8)
    cv2.circle(cmask, (size//2, size//2), int(size//2 * 0.97), 255, -1)
    img[cmask == 0] = 0

    # 5. CLAHE on LAB L-channel
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    lab[:, :, 0] = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)).apply(lab[:, :, 0])
    img = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

    # 6. Ben Graham sharpening
    sigma = max((size // sigma_ratio) | 1, 1)
    blur  = cv2.GaussianBlur(img, (0, 0), sigmaX=sigma)
    img   = cv2.addWeighted(img, 4, blur, -4, 128)
    img[cmask == 0] = 0

    return img

# Latency check
t = time.time()
for _ in range(3):
    preprocess_fundus(df["path"].iloc[0], size=512)
print(f"✅ preprocess_fundus defined  |  latency: {(time.time()-t)/3*1e3:.1f} ms/img")


## 🔀 Cell 5 — Augmentation Pipelines

In [ ]:
_A_NEW = Version(A.__version__) >= Version("1.4.0")

def _gauss_noise():
    return (A.GaussNoise(std_range=(0.03, 0.12), p=0.2) if _A_NEW
            else A.GaussNoise(var_limit=(10.0, 50.0), p=0.2))

def _coarse_dropout(img_size):
    h = img_size // 16
    return (A.CoarseDropout(num_holes_range=(1,8), hole_height_range=(h,h),
                            hole_width_range=(h,h), p=0.2) if _A_NEW
            else A.CoarseDropout(max_holes=8, max_height=h, max_width=h, p=0.2))

def get_train_transform(img_size):
    return A.Compose([
        A.RandomResizedCrop(height=img_size, width=img_size,
                            scale=(0.8, 1.0), ratio=(0.9, 1.1)),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1,
                           rotate_limit=15, p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.15,
                                   contrast_limit=0.15, p=0.4),
        A.CLAHE(clip_limit=2.0, p=0.2),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20,
                             val_shift_limit=10, p=0.3),
        A.RandomGamma(gamma_limit=(80, 120), p=0.3),
        _gauss_noise(),
        A.MotionBlur(blur_limit=3, p=0.1),
        _coarse_dropout(img_size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

def get_val_transform(img_size):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

# 5-view TTA
def get_tta_transforms(img_size):
    base = [A.Resize(img_size, img_size),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()]
    return [
        A.Compose(base),
        A.Compose([A.HorizontalFlip(p=1.0)] + base),
        A.Compose([A.VerticalFlip(p=1.0)]   + base),
        A.Compose([A.Rotate(limit=10, p=1.0)] + base),
        A.Compose([A.Rotate(limit=-10, p=1.0)] + base),
    ]

print(f"✅ Albumentations {A.__version__} | API mode: {'new ≥1.4' if _A_NEW else 'legacy <1.4'}")


## 📚 Cell 6 — Dataset Class

In [ ]:
class AptosDataset(Dataset):
    """
    General APTOS dataset.
    label_mode:
      "regression" → float label in [0,4]  (used by regression head)
      "class"      → int label              (used by 2-stage heads)
    """
    def __init__(self, df, transform=None, label_mode="regression",
                 preprocess=True, img_size=512):
        self.df         = df.reset_index(drop=True)
        self.transform  = transform
        self.label_mode = label_mode
        self.preprocess = preprocess
        self.img_size   = img_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = (preprocess_fundus(row["path"], size=self.img_size)
               if self.preprocess else self._raw(row["path"]))

        if self.transform:
            img = self.transform(image=img)["image"]
        else:
            img = torch.from_numpy(img.transpose(2,0,1)).float() / 255.0

        grade = int(row["diagnosis"])
        label = (torch.tensor(float(grade), dtype=torch.float32)
                 if self.label_mode == "regression"
                 else torch.tensor(grade, dtype=torch.long))
        return img, label

    def _raw(self, path):
        bgr = cv2.imread(str(path))
        img = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        return cv2.resize(img, (self.img_size, self.img_size))


class OrdinalDataset(Dataset):
    """Stage-2 ordinal: DR-positive only (grade 1-4). Cumulative binary encoding."""
    ORDINAL = {1:[1,0,0,0], 2:[1,1,0,0], 3:[1,1,1,0], 4:[1,1,1,1]}

    def __init__(self, df, transform=None, img_size=384):
        self.df        = df[df.diagnosis >= 1].reset_index(drop=True)
        self.transform = transform
        self.img_size  = img_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        grade = int(row["diagnosis"])
        img   = preprocess_fundus(row["path"], size=self.img_size)
        if self.transform:
            img = self.transform(image=img)["image"]
        return img, torch.tensor(self.ORDINAL[grade], dtype=torch.float32)


def make_weighted_loader(df_split, dataset, batch_size, drop_last=False):
    """DataLoader with WeightedRandomSampler to handle class imbalance."""
    labels  = df_split["diagnosis"].values.astype(int)
    counts  = np.bincount(labels, minlength=NUM_CLASSES).astype(float)
    w_class = 1.0 / np.maximum(counts, 1)
    s_wts   = torch.tensor([w_class[l] for l in labels], dtype=torch.float)
    sampler = WeightedRandomSampler(s_wts, num_samples=len(s_wts), replacement=True)
    nw = min(4, os.cpu_count() or 1)
    return DataLoader(dataset, batch_size=batch_size, sampler=sampler,
                      num_workers=nw, pin_memory=(DEVICE.type=="cuda"),
                      drop_last=drop_last, persistent_workers=(nw>0))

def make_loader(dataset, batch_size, shuffle=False, drop_last=False):
    nw = min(4, os.cpu_count() or 1)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                      num_workers=nw, pin_memory=(DEVICE.type=="cuda"),
                      drop_last=drop_last, persistent_workers=(nw>0))

print("✅ AptosDataset, OrdinalDataset, make_weighted_loader, make_loader defined.")


## 🏗️ Cell 7 — Model Architecture
**Regression mode** (single float output, range [0,4]) for main K-Fold training.  
**Classification mode** (1 or 4 outputs) for optional 2-stage pipeline.


In [ ]:
class DRModel(nn.Module):
    """
    Unified DR model.
    mode="regression"  → 1 float output    (MSE loss, OptimizedRounder)
    mode="binary"      → 1 sigmoid output  (Stage 1)
    mode="ordinal"     → 4 sigmoid outputs (Stage 2)
    mode="classify"    → 5 softmax outputs (direct classifier)
    """
    def __init__(self, model_name=CFG["model_name"], mode="regression",
                 pretrained=True, dropout=0.5):
        super().__init__()
        self.mode = mode
        out_map   = {"regression":1, "binary":1, "ordinal":4, "classify":5}
        n_out     = out_map[mode]

        self.backbone = timm.create_model(
            model_name, pretrained=pretrained,
            num_classes=0, global_pool="avg"
        )
        feat = self.backbone.num_features
        self.head = nn.Sequential(
            nn.BatchNorm1d(feat),
            nn.Linear(feat, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, n_out),
        )

    def forward(self, x):
        out = self.head(self.backbone(x))
        return out.squeeze(-1) if self.mode in ("regression","binary") else out

    def freeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad_(False)

    def unfreeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad_(True)

    def unfreeze_top(self, n=4):
        self.freeze_backbone()
        if hasattr(self.backbone, "blocks"):
            for blk in list(self.backbone.blocks)[-n:]:
                for p in blk.parameters():
                    p.requires_grad_(True)
        for attr in ("conv_head","bn2","norm_head","norm"):
            if hasattr(self.backbone, attr):
                for p in getattr(self.backbone, attr).parameters():
                    p.requires_grad_(True)

# Sanity check
_m = DRModel(mode="regression", pretrained=False).to(DEVICE)
_x = torch.randn(2, 3, 224, 224).to(DEVICE)
_o = _m(_x)
print(f"✅ DRModel regression: input {list(_x.shape)} → output {list(_o.shape)}")
del _m, _x, _o; gc.collect()


## ⚖️ Cell 8 — Loss Functions, OptimizedRounder & Metrics

**Why regression + rounding beats direct classification for QWK:**  
QWK penalises distant misclassifications quadratically.  
A regression model predicting continuous grades naturally minimises that penalty,  
while a classifier treats all wrong classes equally.  
The `OptimizedRounder` then fits the 4 cut-points to maximize QWK directly.


In [ ]:
# ── Regression losses ─────────────────────────────────────────────────────────
class MSELoss(nn.Module):
    """Standard MSE — primary loss for regression approach."""
    def forward(self, pred, target):
        return F.mse_loss(pred, target)

class SmoothL1Loss(nn.Module):
    """Smooth L1 — less sensitive to outlier labels."""
    def __init__(self, beta=1.0):
        super().__init__(); self.beta = beta
    def forward(self, pred, target):
        return F.smooth_l1_loss(pred, target, beta=self.beta)

class HybridRegressionLoss(nn.Module):
    """0.5 × MSE + 0.5 × SmoothL1 — balances precision and robustness."""
    def __init__(self):
        super().__init__()
        self.mse  = MSELoss()
        self.sl1  = SmoothL1Loss(beta=1.0)
    def forward(self, pred, target):
        return 0.5 * self.mse(pred, target) + 0.5 * self.sl1(pred, target)

# ── Focal & Binary losses (2-stage) ──────────────────────────────────────────
class BinaryFocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__(); self.alpha=alpha; self.gamma=gamma
    def forward(self, logits, targets):
        bce   = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        prob  = torch.sigmoid(logits)
        p_t   = prob*targets + (1-prob)*(1-targets)
        a_t   = self.alpha*targets + (1-self.alpha)*(1-targets)
        return (a_t * (1-p_t)**self.gamma * bce).mean()

# ── OptimizedRounder (Nelder-Mead) ────────────────────────────────────────────
class OptimizedRounder:
    """
    Fits 4 threshold cut-points [t0,t1,t2,t3] on continuous regression outputs
    to maximise QWK.  Standard technique in Kaggle ordinal competitions.
    """
    def __init__(self):
        self.coef_ = np.array([0.5, 1.5, 2.5, 3.5])

    def _qwk_loss(self, coef, X, y):
        cuts  = np.sort(coef)
        preds = pd.cut(X, bins=[-np.inf] + list(cuts) + [np.inf],
                       labels=[0,1,2,3,4]).astype(int)
        return -cohen_kappa_score(y, preds, weights="quadratic")

    def fit(self, X, y):
        loss_fn = lambda c: self._qwk_loss(c, X, y)
        result  = minimize(loss_fn, self.coef_, method="Nelder-Mead",
                           options={"maxiter":2000, "xatol":1e-6, "fatol":1e-9})
        self.coef_ = np.sort(result.x)
        return self

    def predict(self, X):
        return pd.cut(X, bins=[-np.inf] + list(self.coef_) + [np.inf],
                      labels=[0,1,2,3,4]).astype(int).values

    def predict_clipped(self, X):
        """predict + clip to [0,4] for safety."""
        return np.clip(self.predict(X), 0, 4)

# ── Metric helpers ────────────────────────────────────────────────────────────
def qwk(y_true, y_pred):
    return cohen_kappa_score(np.array(y_true), np.array(y_pred), weights="quadratic")

def compute_class_weights(labels, n_classes=5):
    counts  = np.bincount(labels, minlength=n_classes).astype(float)
    weights = len(labels) / (n_classes * np.maximum(counts, 1))
    return torch.tensor(weights / weights.sum() * n_classes, dtype=torch.float32)

print("✅ HybridRegressionLoss, BinaryFocalLoss, OptimizedRounder, qwk() defined.")


## 📊 Cell 9 — EDA: Class Distribution

In [ ]:
_eda = PLOT_DIR / "eda_distribution.png"
if _eda.exists():
    print("✅ [RESUME] EDA already done.")
    plt.imshow(plt.imread(str(_eda))); plt.axis("off"); plt.tight_layout(); plt.show()
else:
    counts = [int((df.diagnosis==g).sum()) for g in range(5)]
    labels = [f"G{g}\n{GRADE_MAP[g]}" for g in range(5)]
    fig, axes = plt.subplots(1, 2, figsize=(14,5))

    axes[0].bar(labels, counts, color=GRADE_COLORS, edgecolor="black", lw=0.6)
    axes[0].set_title("Grade Distribution (Absolute)", fontsize=13, fontweight="bold")
    for bar, n in zip(axes[0].patches, counts):
        axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+15,
                     str(n), ha="center", fontsize=10)

    axes[1].pie([c/sum(counts)*100 for c in counts], labels=labels,
                colors=GRADE_COLORS, autopct="%1.1f%%", startangle=140)
    axes[1].set_title("Grade Distribution (%)", fontsize=13, fontweight="bold")

    plt.tight_layout()
    plt.savefig(str(_eda), dpi=120, bbox_inches="tight")
    plt.show()
    print(f"✅ EDA saved → {_eda}")


## ✂️ Cell 10 — Stratified 5-Fold Split

In [ ]:
_splits = ARTIFACT_DIR / "kfold_splits.parquet"
if _splits.exists():
    df = pd.read_parquet(_splits)
    print("✅ [RESUME] K-Fold splits loaded.")
else:
    skf = StratifiedKFold(n_splits=CFG["n_folds"], shuffle=True, random_state=CFG["seed"])
    df["fold"] = -1
    for fi, (_, vi) in enumerate(skf.split(df, df.diagnosis)):
        df.loc[vi, "fold"] = fi
    df.to_parquet(_splits, index=False)
    print("✅ 5-Fold splits created.")

print("\nFold distribution:")
for f in range(CFG["n_folds"]):
    n  = (df.fold==f).sum()
    gd = df[df.fold==f].diagnosis.value_counts().sort_index()
    gs = " | ".join(f"G{g}:{c}" for g,c in gd.items())
    print(f"  Fold {f}: {n:5d}  [{gs}]")


## 🏋️ Cell 11 — Trainer Class + K-Fold Training
**Resume-safe**: completed folds are detected by checkpoint flag files and skipped.  
Each fold runs 3 phases of progressive resizing with early stopping per phase.


In [ ]:
# ── Checkpoint helpers ────────────────────────────────────────────────────────
def save_ckpt(path, model, optimizer, scheduler, scaler, epoch, batch,
              phase_id, best_qwk, rounder_coef):
    torch.save({
        "model"       : model.state_dict(),
        "optimizer"   : optimizer.state_dict(),
        "scheduler"   : scheduler.state_dict(),
        "scaler"      : scaler.state_dict() if scaler else None,
        "epoch"       : epoch,
        "batch"       : batch,
        "phase_id"    : phase_id,
        "best_qwk"    : best_qwk,
        "rounder_coef": rounder_coef,
    }, path)

def load_ckpt(path, model, optimizer=None, scheduler=None, scaler=None,
              map_location="cpu"):
    ckpt = safe_load(path, map_location)
    model.load_state_dict(ckpt["model"])
    if optimizer  and ckpt.get("optimizer"):  optimizer.load_state_dict(ckpt["optimizer"])
    if scheduler  and ckpt.get("scheduler"):  scheduler.load_state_dict(ckpt["scheduler"])
    if scaler     and ckpt.get("scaler"):      scaler.load_state_dict(ckpt["scaler"])
    return ckpt


# ── Trainer ───────────────────────────────────────────────────────────────────
class Trainer:
    def __init__(self, model, fold, scaler=None):
        self.model    = model
        self.fold     = fold
        self.scaler   = scaler
        self.best_qwk = -1.0
        self.rounder  = OptimizedRounder()

    def train_epoch(self, loader, optimizer, criterion, scheduler,
                    start_batch=0, device=DEVICE):
        self.model.train()
        total_loss = 0.0
        pbar = tqdm(enumerate(loader), total=len(loader),
                    desc="  train", leave=False)
        for i, (imgs, labels) in pbar:
            if i < start_batch:
                scheduler.step(); continue
            imgs   = imgs.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()

            if self.scaler is not None:
                with torch.amp.autocast("cuda"):
                    preds = self.model(imgs)
                    loss  = criterion(preds, labels)
                self.scaler.scale(loss).backward()
                self.scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    self.model.parameters(), CFG["grad_clip"])
                self.scaler.step(optimizer)
                self.scaler.update()
            else:
                preds = self.model(imgs)
                loss  = criterion(preds, labels)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    self.model.parameters(), CFG["grad_clip"])
                optimizer.step()

            scheduler.step()
            total_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        return total_loss / max(len(loader) - start_batch, 1)

    @torch.no_grad()
    def validate(self, loader, fit_rounder=True, device=DEVICE):
        self.model.eval()
        all_preds, all_labels = [], []
        for imgs, labels in tqdm(loader, desc="  val  ", leave=False):
            imgs = imgs.to(device)
            if self.scaler is not None:
                with torch.amp.autocast("cuda"):
                    preds = self.model(imgs)
            else:
                preds = self.model(imgs)
            all_preds.append(preds.cpu().float().numpy())
            all_labels.append(labels.float().numpy())

        raw_preds  = np.concatenate(all_preds).flatten()
        raw_labels = np.concatenate(all_labels).flatten().astype(int)

        if fit_rounder:
            self.rounder.fit(raw_preds, raw_labels)
        final_preds = self.rounder.predict_clipped(raw_preds)
        val_qwk = qwk(raw_labels, final_preds)
        val_acc = accuracy_score(raw_labels, final_preds)
        return val_qwk, val_acc, raw_preds, raw_labels


# ── K-Fold training loop ──────────────────────────────────────────────────────
fold_best_qwks = []
oof_raw        = np.zeros(len(df), dtype=np.float32)
oof_labels_arr = df.diagnosis.values.copy()
master_rounder = OptimizedRounder()

# Fixed: GradScaler API — use device-aware constructor
if USE_AMP:
    scaler_global = torch.amp.GradScaler("cuda")
else:
    scaler_global = None

print("=" * 65)
print(f"  5-FOLD CV | Regression | Backbone: {CFG['model_name']}")
print("=" * 65)

for fold in range(CFG["n_folds"]):
    ckpt_best = ARTIFACT_DIR / f"fold{fold}_best.pt"
    oof_file  = ARTIFACT_DIR / f"fold{fold}_oof_raw.npy"
    done_flag = ARTIFACT_DIR / f"_done_fold{fold}.flag"

    if done_flag.exists() and ckpt_best.exists():
        prev = safe_load(ckpt_best, "cpu")
        fold_best_qwks.append(prev.get("best_qwk", 0.0))
        if oof_file.exists():
            val_idx = df[df.fold==fold].index
            oof_raw[val_idx] = np.load(str(oof_file))
        print(f"  ✅ [RESUME] Fold {fold} | Best QWK={fold_best_qwks[-1]:.4f}")
        continue

    print(f"\n  ━━━━━━━━━━  FOLD {fold}  ━━━━━━━━━━")
    df_tr = df[df.fold != fold].reset_index(drop=True)
    df_va = df[df.fold == fold].reset_index(drop=True)
    val_idx = df[df.fold == fold].index

    model   = DRModel(mode="regression", pretrained=True).to(DEVICE)
    trainer = Trainer(model, fold, scaler=scaler_global)
    criterion = HybridRegressionLoss()

    fold_best_qwk = -1.0
    best_state    = None
    best_rounder_coef = trainer.rounder.coef_.copy()

    for phase in CFG["phases"]:
        pid   = phase["id"]
        sz    = phase["size"]
        bs    = phase["batch_size"]
        n_ep  = phase["epochs"]
        do_freeze = phase["freeze"]
        print(f"  Phase {pid}: {n_ep} epochs @ {sz}px  "
              f"{'[backbone frozen]' if do_freeze else '[backbone training]'}")

        if do_freeze:
            model.freeze_backbone()
        else:
            if pid == 2:
                model.unfreeze_top(n=4)
            else:
                model.unfreeze_backbone()

        tr_ds = AptosDataset(df_tr, get_train_transform(sz),
                             label_mode="regression", img_size=sz)
        va_ds = AptosDataset(df_va, get_val_transform(sz),
                             label_mode="regression", img_size=sz)
        tr_ld = make_weighted_loader(df_tr, tr_ds, bs, drop_last=True)
        va_ld = make_loader(va_ds, bs)

        lr_phase = CFG["lr"] / (3 ** (pid - 1))
        optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=lr_phase, weight_decay=CFG["weight_decay"])
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer, max_lr=lr_phase,
            steps_per_epoch=len(tr_ld), epochs=n_ep,
            pct_start=0.2, anneal_strategy="cos",
            div_factor=10, final_div_factor=100)

        patience_cnt = 0
        for ep in range(n_ep):
            tr_loss = trainer.train_epoch(tr_ld, optimizer, criterion, scheduler)
            val_qwk_, val_acc_, _, _ = trainer.validate(va_ld, fit_rounder=True)

            improved = val_qwk_ > fold_best_qwk
            if improved:
                fold_best_qwk     = val_qwk_
                best_state        = deepcopy(model.state_dict())
                best_rounder_coef = trainer.rounder.coef_.copy()
                save_ckpt(ckpt_best, model, optimizer, scheduler,
                          scaler_global, ep, 0, pid, fold_best_qwk,
                          best_rounder_coef)
                patience_cnt = 0
            else:
                patience_cnt += 1

            star = " ★" if improved else ""
            print(f"    P{pid} Ep{ep+1:02d}/{n_ep}: "
                  f"loss={tr_loss:.4f}  QWK={val_qwk_:.4f}  "
                  f"Acc={val_acc_*100:.1f}%{star}")

            if patience_cnt >= CFG["patience"]:
                print(f"    ↳ Early stop (patience={CFG['patience']})")
                break

    # OOF raw predictions with best model
    model.load_state_dict(best_state)
    trainer.rounder.coef_ = best_rounder_coef
    va_ds_final = AptosDataset(df_va, get_val_transform(512),
                               label_mode="regression", img_size=512)
    va_ld_final = make_loader(va_ds_final, 8)
    _, _, fold_raw, _ = trainer.validate(va_ld_final, fit_rounder=False)

    oof_raw[val_idx] = fold_raw[:len(val_idx)]
    np.save(str(oof_file), fold_raw[:len(val_idx)])
    fold_best_qwks.append(fold_best_qwk)
    done_flag.touch()
    print(f"  ✅ Fold {fold} complete — Best QWK: {fold_best_qwk:.4f}")
    del model; gc.collect()
    if DEVICE.type=="cuda": torch.cuda.empty_cache()

np.save(str(ARTIFACT_DIR/"oof_raw.npy"),    oof_raw)
np.save(str(ARTIFACT_DIR/"oof_labels.npy"), oof_labels_arr)

# Fit global rounder on all OOF raw outputs
master_rounder.fit(oof_raw, oof_labels_arr)
oof_preds_rounded = master_rounder.predict_clipped(oof_raw)
oof_qwk_val = qwk(oof_labels_arr, oof_preds_rounded)
oof_acc_val = accuracy_score(oof_labels_arr, oof_preds_rounded)

np.save(str(ARTIFACT_DIR/"oof_preds.npy"), oof_preds_rounded)
np.save(str(ARTIFACT_DIR/"rounder_coef.npy"), master_rounder.coef_)
st_save("oof_qwk", float(oof_qwk_val))
st_save("oof_acc", float(oof_acc_val))

print("\n" + "="*65)
for i,q in enumerate(fold_best_qwks):
    print(f"  Fold {i}: QWK = {q:.4f}")
print(f"  Mean : {np.mean(fold_best_qwks):.4f} ± {np.std(fold_best_qwks):.4f}")
print(f"  OOF  QWK = {oof_qwk_val:.4f}   Acc = {oof_acc_val*100:.2f}%")
print(f"  Rounder coefs: {np.round(master_rounder.coef_, 3)}")
print("="*65)


## 📈 Cell 12 — OOF Confusion Matrix & Per-Class Metrics

In [ ]:
oof_raw_l    = np.load(str(ARTIFACT_DIR/"oof_raw.npy"))
oof_labels_l = np.load(str(ARTIFACT_DIR/"oof_labels.npy"))
oof_preds_l  = np.load(str(ARTIFACT_DIR/"oof_preds.npy"))
_coef        = np.load(str(ARTIFACT_DIR/"rounder_coef.npy"))

oof_qwk_disp = qwk(oof_labels_l, oof_preds_l)
oof_acc_disp = accuracy_score(oof_labels_l, oof_preds_l)

cm   = confusion_matrix(oof_labels_l, oof_preds_l)
fig, axes = plt.subplots(1, 2, figsize=(16,6))

disp = ConfusionMatrixDisplay(cm, display_labels=[f"G{i}" for i in range(5)])
disp.plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title(f"OOF Confusion Matrix\nQWK={oof_qwk_disp:.4f}  Acc={oof_acc_disp*100:.1f}%",
                  fontweight="bold")

report = classification_report(oof_labels_l, oof_preds_l,
    target_names=[f"G{i} {GRADE_MAP[i]}" for i in range(5)], output_dict=True)
recalls    = [report[f"G{i} {GRADE_MAP[i]}"]["recall"]    for i in range(5)]
precisions = [report[f"G{i} {GRADE_MAP[i]}"]["precision"] for i in range(5)]
x = np.arange(5); w = 0.35
axes[1].bar(x-w/2, recalls,    width=w, color=GRADE_COLORS, label="Recall",    alpha=0.9)
axes[1].bar(x+w/2, precisions, width=w, color=GRADE_COLORS, label="Precision", alpha=0.5, hatch="//")
axes[1].set_xticks(x); axes[1].set_xticklabels([f"G{i}" for i in range(5)])
axes[1].set_ylim(0,1.1); axes[1].legend()
axes[1].set_title("Per-Class Recall & Precision", fontweight="bold")

plt.tight_layout()
plt.savefig(str(PLOT_DIR/"oof_confusion_matrix.png"), dpi=120, bbox_inches="tight")
plt.show()
print(classification_report(oof_labels_l, oof_preds_l,
      target_names=[f"G{i} {GRADE_MAP[i]}" for i in range(5)]))

try:
    from sklearn.preprocessing import label_binarize
    y_bin = label_binarize(oof_labels_l, classes=list(range(5)))
    raw_norm = (oof_raw_l - oof_raw_l.min()) / (np.ptp(oof_raw_l) + 1e-8)  # fixed: .ptp() → np.ptp()
    print(f"Rounder thresholds: {np.round(_coef, 3)}")
except Exception as e:
    print(f"AUROC skip: {e}")


## 🤝 Cell 13 — Ensemble Inference (5-Fold × 5 TTA)
Uses all fold models on fold-0 held-out test set.

In [ ]:
TEST_FOLD = 0
df_test   = df[df.fold == TEST_FOLD].reset_index(drop=True)
print(f"Ensemble inference on {len(df_test)} held-out samples ...")

# Load global rounder
_coef_global     = np.load(str(ARTIFACT_DIR/"rounder_coef.npy"))
ensemble_rounder = OptimizedRounder()
ensemble_rounder.coef_ = _coef_global

ensemble_raw = np.zeros(len(df_test), dtype=np.float32)
n_models     = 0

for fold in range(1, CFG["n_folds"]):
    ckpt_path = ARTIFACT_DIR / f"fold{fold}_best.pt"
    if not ckpt_path.exists():
        print(f"  ⚠️  fold{fold}_best.pt not found — skipping"); continue

    ckpt  = safe_load(ckpt_path, DEVICE)
    model = DRModel(mode="regression", pretrained=False).to(DEVICE)
    model.load_state_dict(ckpt["model"]); model.eval()

    fold_rounder = OptimizedRounder()
    fold_coef    = ckpt.get("rounder_coef", _coef_global)
    fold_rounder.coef_ = np.array(fold_coef)

    fold_raw = np.zeros(len(df_test), dtype=np.float32)
    tta_list = get_tta_transforms(512)

    with torch.no_grad():
        for tf in tta_list:
            ds  = AptosDataset(df_test, tf, label_mode="regression", img_size=512)
            ld  = make_loader(ds, 8)
            bp  = []
            for imgs, _ in tqdm(ld, desc=f"  fold{fold} TTA", leave=False):
                if USE_AMP:
                    with torch.amp.autocast("cuda"):
                        preds = model(imgs.to(DEVICE))
                else:
                    preds = model(imgs.to(DEVICE))
                bp.extend(preds.cpu().float().numpy().tolist())
            fold_raw += np.array(bp[:len(df_test)])

    fold_raw /= len(tta_list)
    ensemble_raw += fold_raw
    n_models += 1

    fp_preds = fold_rounder.predict_clipped(fold_raw)
    fp_qwk   = qwk(df_test.diagnosis.values, fp_preds)
    print(f"  Fold {fold}: QWK={fp_qwk:.4f}  (rounder={np.round(fold_rounder.coef_,2)})")
    del model; gc.collect()
    if DEVICE.type=="cuda": torch.cuda.empty_cache()

if n_models > 0:
    ensemble_raw /= n_models
    ens_rounder = OptimizedRounder()
    ens_rounder.fit(ensemble_raw, df_test.diagnosis.values)
    test_preds   = ens_rounder.predict_clipped(ensemble_raw)
    test_labels  = df_test.diagnosis.values
    test_qwk_val = qwk(test_labels, test_preds)
    test_acc_val = accuracy_score(test_labels, test_preds)

    print(f"\n  ✅ Ensemble ({n_models} models × {len(tta_list)} TTA)")
    print(f"     Test QWK : {test_qwk_val:.4f}")
    print(f"     Test Acc : {test_acc_val*100:.2f}%")
    print(f"     Thresholds: {np.round(ens_rounder.coef_, 3)}")
    st_save("test_qwk",  float(test_qwk_val))
    st_save("test_acc",  float(test_acc_val))
    np.save(str(ARTIFACT_DIR/"ensemble_test_raw.npy"),  ensemble_raw)
    np.save(str(ARTIFACT_DIR/"ensemble_test_preds.npy"), test_preds)
else:
    print("⚠️  No fold models found — run Cell 11 first.")


## 🧠 Cell 14 — Optional 2-Stage Pipeline
Complements the regression approach for cases where Stage-1 binary confidence is needed.  
**Stage 1**: No DR (0) vs DR (1-4) — BinaryFocalLoss  
**Stage 2**: Ordinal grades 1-4 — per-node BCE


In [ ]:
df_tr2 = df[df.fold != 0].reset_index(drop=True)
df_va2 = df[df.fold == 0].reset_index(drop=True)

S1_CKPT = ARTIFACT_DIR / "stage1_best.pt"
S2_CKPT = ARTIFACT_DIR / "stage2_best.pt"

class BinaryDataset(Dataset):
    def __init__(self, df, transform=None, img_size=384):
        self.df=df.reset_index(drop=True); self.tf=transform; self.sz=img_size
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        img=preprocess_fundus(row["path"], size=self.sz)
        if self.tf: img=self.tf(image=img)["image"]
        return img, torch.tensor(float(row.diagnosis>=1), dtype=torch.float32)

class OrdinalDatasetS2(Dataset):
    ORD={1:[1,0,0,0],2:[1,1,0,0],3:[1,1,1,0],4:[1,1,1,1]}
    def __init__(self, df, transform=None, img_size=384):
        self.df=df[df.diagnosis>=1].reset_index(drop=True)
        self.tf=transform; self.sz=img_size
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        img=preprocess_fundus(row["path"], size=self.sz)
        if self.tf: img=self.tf(image=img)["image"]
        return img, torch.tensor(self.ORD[int(row.diagnosis)], dtype=torch.float32)

def _train_binary(df_tr, df_va, epochs=15, img_sz=384):
    BS=16
    scl = torch.amp.GradScaler("cuda") if USE_AMP else None
    tr_ds=BinaryDataset(df_tr, get_train_transform(img_sz), img_sz)
    va_ds=BinaryDataset(df_va, get_val_transform(img_sz),   img_sz)
    tr_ld=make_loader(tr_ds, BS, shuffle=True, drop_last=True)
    va_ld=make_loader(va_ds, BS)
    m=DRModel(mode="binary", pretrained=True).to(DEVICE)
    m.freeze_backbone()
    crit=BinaryFocalLoss()
    opt=torch.optim.AdamW(filter(lambda p:p.requires_grad,m.parameters()),
                          lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    best_loss=float("inf"); best_state=None
    for ep in range(epochs):
        m.train(); ep_loss=0
        for imgs,labels in tqdm(tr_ld,desc=f"  S1 ep{ep+1}",leave=False):
            imgs=imgs.to(DEVICE); labels=labels.to(DEVICE); opt.zero_grad()
            if scl:
                with torch.amp.autocast("cuda"): loss=crit(m(imgs),labels)
                scl.scale(loss).backward(); scl.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(m.parameters(),1.0)
                scl.step(opt); scl.update()
            else:
                loss=crit(m(imgs),labels); loss.backward()
                torch.nn.utils.clip_grad_norm_(m.parameters(),1.0); opt.step()
            sched.step(); ep_loss+=loss.item()
        vl=ep_loss/len(tr_ld)
        if vl<best_loss: best_loss=vl; best_state=deepcopy(m.state_dict())
        print(f"    ep{ep+1:02d}: loss={vl:.4f}")
        if ep==epochs//2: m.unfreeze_top(4)
    return best_state, best_loss

def _train_ordinal(df_tr, df_va, epochs=15, img_sz=384):
    BS=16
    scl = torch.amp.GradScaler("cuda") if USE_AMP else None
    tr_ds=OrdinalDatasetS2(df_tr, get_train_transform(img_sz), img_sz)
    va_ds=OrdinalDatasetS2(df_va, get_val_transform(img_sz),   img_sz)
    tr_ld=make_loader(tr_ds, BS, shuffle=True, drop_last=True)
    va_ld=make_loader(va_ds, BS)
    m=DRModel(mode="ordinal", pretrained=True).to(DEVICE)
    m.freeze_backbone()
    crit=nn.BCEWithLogitsLoss()
    opt=torch.optim.AdamW(filter(lambda p:p.requires_grad,m.parameters()),
                          lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    best_loss=float("inf"); best_state=None
    for ep in range(epochs):
        m.train(); ep_loss=0
        for imgs,labels in tqdm(tr_ld,desc=f"  S2 ep{ep+1}",leave=False):
            imgs=imgs.to(DEVICE); labels=labels.to(DEVICE); opt.zero_grad()
            if scl:
                with torch.amp.autocast("cuda"): loss=crit(m(imgs),labels)
                scl.scale(loss).backward(); scl.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(m.parameters(),1.0)
                scl.step(opt); scl.update()
            else:
                loss=crit(m(imgs),labels); loss.backward()
                torch.nn.utils.clip_grad_norm_(m.parameters(),1.0); opt.step()
            sched.step(); ep_loss+=loss.item()
        vl=ep_loss/len(tr_ld)
        if vl<best_loss: best_loss=vl; best_state=deepcopy(m.state_dict())
        print(f"    ep{ep+1:02d}: loss={vl:.4f}")
        if ep==epochs//2: m.unfreeze_top(4)
    return best_state, best_loss

# Train Stage 1
if (ARTIFACT_DIR/"_done_s1.flag").exists() and S1_CKPT.exists():
    print("✅ [RESUME] Stage 1 already trained.")
else:
    print("━━━━  STAGE 1: Binary (No DR vs DR)  ━━━━")
    s1_state, s1_loss = _train_binary(df_tr2, df_va2)
    torch.save({"model":s1_state,"loss":s1_loss}, S1_CKPT)
    (ARTIFACT_DIR/"_done_s1.flag").touch()
    print(f"  ✅ Stage 1 saved | val_loss={s1_loss:.4f}")

# Train Stage 2
if (ARTIFACT_DIR/"_done_s2.flag").exists() and S2_CKPT.exists():
    print("✅ [RESUME] Stage 2 already trained.")
else:
    print("━━━━  STAGE 2: Ordinal (Grades 1-4)  ━━━━")
    s2_state, s2_loss = _train_ordinal(df_tr2, df_va2)
    torch.save({"model":s2_state,"loss":s2_loss}, S2_CKPT)
    (ARTIFACT_DIR/"_done_s2.flag").touch()
    print(f"  ✅ Stage 2 saved | val_loss={s2_loss:.4f}")

# ── 2-Stage inference ─────────────────────────────────────────────────────────
@torch.no_grad()
def two_stage_predict(paths, thresh=0.5, tta=True, device=DEVICE):
    s1=DRModel(mode="binary",  pretrained=False).to(device)
    s2=DRModel(mode="ordinal", pretrained=False).to(device)
    s1.load_state_dict(safe_load(S1_CKPT, device)["model"]); s1.eval()
    s2.load_state_dict(safe_load(S2_CKPT, device)["model"]); s2.eval()
    df_inf=pd.DataFrame({"path":paths,"diagnosis":[0]*len(paths)})
    tfs=get_tta_transforms(384) if tta else [get_val_transform(384)]

    s1p=np.zeros(len(df_inf),dtype=np.float32)
    for tf in tfs:
        ds=BinaryDataset(df_inf,tf,384); ld=make_loader(ds,8)
        bp=[]
        for imgs,_ in ld:
            if USE_AMP:
                with torch.amp.autocast("cuda"): o=s1(imgs.to(device))
            else: o=s1(imgs.to(device))
            p=torch.sigmoid(o).squeeze()
            bp.extend((p.cpu().numpy() if p.dim()>0 else [float(p.cpu())]))
        s1p+=np.array(bp[:len(df_inf)])
    s1p/=len(tfs); s1bin=(s1p>=thresh).astype(int)

    dr_idx=np.where(s1bin==1)[0]; grades=np.zeros(len(df_inf),dtype=int)
    if len(dr_idx)>0:
        df_dr=df_inf.iloc[dr_idx].reset_index(drop=True)
        s2n=np.zeros((len(df_dr),4),dtype=np.float32)
        for tf in tfs:
            ds=OrdinalDatasetS2(df_dr,tf,384); ld=make_loader(ds,8)
            bp=[]
            for imgs,_ in ld:
                if USE_AMP:
                    with torch.amp.autocast("cuda"): o=s2(imgs.to(device))
                else: o=s2(imgs.to(device))
                bp.append(torch.sigmoid(o).cpu().numpy())
            s2n+=np.concatenate(bp,axis=0)[:len(df_dr)]
        s2n/=len(tfs)
        grades[dr_idx]=np.clip((s2n>=0.5).sum(axis=1)+1,1,4)
    del s1,s2; gc.collect()
    return grades, s1p

print("\nEvaluating 2-stage on val fold 0 ...")
_paths=df_va2.path.tolist(); _true=df_va2.diagnosis.values
_pred, _s1p = two_stage_predict(_paths)
_ts_qwk=qwk(_true,_pred)
_ts_acc=accuracy_score(_true,_pred)
print(f"  2-Stage QWK : {_ts_qwk:.4f}")
print(f"  2-Stage Acc : {_ts_acc*100:.2f}%")
st_save("two_stage_qwk", float(_ts_qwk))
st_save("two_stage_acc", float(_ts_acc))


## 🎨 Cell 15 — Grad-CAM++ Explainability

In [ ]:
try:
    from pytorch_grad_cam import GradCAMPlusPlus                      # correct import
    from pytorch_grad_cam.utils.image import show_cam_on_image
    from pytorch_grad_cam.utils.model_targets import RawScoresOutputTarget

    _bfi = int(np.argmax(fold_best_qwks)) if fold_best_qwks else 0
    _ckpt = safe_load(ARTIFACT_DIR/f"fold{_bfi}_best.pt", DEVICE)
    _m = DRModel(mode="regression", pretrained=False).to(DEVICE)
    _m.load_state_dict(_ckpt["model"]); _m.eval()

    # Robust target-layer detection for EfficientNet / ViT variants
    if hasattr(_m.backbone, "blocks"):
        _tgt_layers = [_m.backbone.blocks[-1][-1]]
    else:
        _last = list(_m.backbone.children())[-1]
        _tgt_layers = [_last if isinstance(_last, nn.Module) else list(_m.backbone.children())[-2]]

    cam = GradCAMPlusPlus(model=_m, target_layers=_tgt_layers)

    fig, axes = plt.subplots(2, 5, figsize=(22,8))
    for grade in range(5):
        sample = df[df.diagnosis==grade].sample(1, random_state=42).iloc[0]
        raw    = preprocess_fundus(sample.path, size=512)
        tf     = get_val_transform(512)
        tensor = tf(image=raw)["image"].unsqueeze(0).to(DEVICE)

        grayscale_cam = cam(input_tensor=tensor, targets=None)[0]
        vis = show_cam_on_image(raw.astype(np.float32)/255.0,
                                grayscale_cam, use_rgb=True)

        axes[0][grade].imshow(raw)
        axes[0][grade].set_title(f"Grade {grade}: {GRADE_MAP[grade]}", fontsize=9)
        axes[0][grade].axis("off")
        axes[1][grade].imshow(vis)
        axes[1][grade].set_title("Grad-CAM++", fontsize=9)
        axes[1][grade].axis("off")

    plt.suptitle("Grad-CAM++ Attention — All 5 DR Grades", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(str(PLOT_DIR/"gradcam_grades.png"), dpi=120, bbox_inches="tight")
    plt.show()
    del _m, cam; gc.collect()
    print("✅ Grad-CAM++ complete.")
except ImportError:
    print("⚠️  pytorch-grad-cam not installed — run Cell 1 first.")
except Exception as e:
    print(f"⚠️  Grad-CAM error: {e}")


## 📋 Cell 16 — Final Results Summary

In [ ]:
state = st_load()
W = 65
print("="*W)
print("  DIABETIC RETINOPATHY GRADING — v18 FINAL SUMMARY")
print("="*W)
print(f"  Backbone     : {CFG['model_name']}")
print(f"  Mode         : Regression + OptimizedRounder")
print(f"  Device       : {DEVICE} | AMP: {'ON' if USE_AMP else 'OFF'}")
print(f"  Dataset      : APTOS 2019 — {len(df):,} images (5 classes)")
print()
print("  ─── K-Fold Cross Validation ───")
if "fold_best_qwks" in dir() and fold_best_qwks:
    for i, q in enumerate(fold_best_qwks):
        print(f"    Fold {i}: QWK = {q:.4f}")
    print(f"    Mean : {np.mean(fold_best_qwks):.4f} ± {np.std(fold_best_qwks):.4f}")
print(f"    OOF QWK  : {state.get('oof_qwk', 'N/A')}")
print(f"    OOF Acc  : {float(state.get('oof_acc',0))*100:.2f}%")
print()
print("  ─── Ensemble (fold-0 test) ───")
print(f"    Test QWK  : {state.get('test_qwk','N/A')}")
print(f"    Test Acc  : {float(state.get('test_acc',0))*100:.2f}%")
print()
print("  ─── 2-Stage Pipeline (fold-0 val) ───")
print(f"    2-Stage QWK : {state.get('two_stage_qwk','N/A')}")
print(f"    2-Stage Acc : {float(state.get('two_stage_acc',0))*100:.2f}%")
print()
print("  ─── Saved Artifacts ───")
for f in sorted(ARTIFACT_DIR.glob("*.pt")):
    print(f"    {f.name:<40s} {f.stat().st_size/1e6:.1f} MB")
print("="*W)
print("  ⚠️  RESEARCH USE ONLY — NOT FOR CLINICAL DEPLOYMENT")
print("="*W)
